# Семинар 13. Две выборки: t-тест, критерий Манна–Уитни, хи-квадрат для категориальных данных

Ноутбук занятия: материал разбирается вместе с преподавателем, ячейки выполняются сверху вниз.

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-13.ipynb`, например `2MP9-Ivanov-Ivan-seminar-13.ipynb`.
- Ноутбук читает файлы `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` из репозитория курса: путь в ячейке задан от папки `notebooks/`, поэтому папки `notebooks/` и `datasets/` должны лежать рядом, как в репозитории. Файлов ноутбук не пишет.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-13-tasks.md` с формулировками и ноутбук `seminar-13-tasks.ipynb` с заготовками и тестами; они решаются на тех же данных, но по Московской области. Ответы вводятся в тест Moodle «Семинар 13».
- Шпаргалка по функциям: `reference/statistics.md`, раздел «Две и больше выборок». На семинаре 14 — контрольная работа № 2 и сдача этапа 3 проекта, `project/03-hypotheses.md`; гипотезы 3 и 4 этапа — сравнение двух групп и связь двух категорий — разбираются сегодня.

Что сегодня. На семинаре 12 одна выборка сравнивалась с известным числом. Сегодня известного числа нет: части данных сравниваются друг с другом. Отличается ли стаж погибших и раненых водителей; что делать, если распределение скошено; как сравнить одни и те же районы в разные годы; что, если групп больше двух; связаны ли два категориальных признака — время суток и гибель людей; значима ли корреляция. Схема прежняя: вопрос, нулевая и альтернативная гипотезы, тест, p-значение, решение, вывод словами. Новое — размер эффекта: рядом с p пишут, насколько велика разница или связь, потому что на тысячах наблюдений значима и ничтожная разница. В конце — таблица шести гипотез как образец раздела отчёта и правила честного вывода.

## 1. Две группы: t-тест Уэлча

Вопрос: отличается ли стаж водителей, погибших в ДТП, от стажа раненых водителей? Обе группы — из одной таблицы, и числа для сравнения извне нет: два средних сравниваются друг с другом.

Подключение — как на семинаре 12, но без statsmodels: все тесты сегодня из `scipy.stats`. Данные те же: водители Брянской области с их ДТП, стаж без кодов 95, 96 и 99, возраст машины — год ДТП минус год выпуска, больше 100 лет — пропуск. Исход ДТП для водителя — столбец `health`: «Погиб», «Ранен», «Не пострадал». Число водителей со стажем, средний и медианный стаж по исходам считает `agg`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')                                         # оформление графиков по умолчанию
dtp = pd.read_csv('../datasets/legal/dtp-bryansk.csv', index_col='id', parse_dates=['datetime'])   # ДТП Брянской области
people = pd.read_csv('../datasets/legal/dtp-bryansk-participants.csv')   # участники ДТП
people['experience_clean'] = people['experience'].replace([95, 96, 99], np.nan)   # коды стажа пропусками
drivers = people[people['role'] == 'Водитель'].merge(dtp, left_on='dtp_id', right_index=True)   # водители с их ДТП
drivers['vehicle_age'] = drivers['datetime'].dt.year - drivers['vehicle_year']   # возраст машины в годах
drivers['vehicle_age'] = drivers['vehicle_age'].where(drivers['vehicle_age'] <= 100)   # невозможный возраст в пропуск
drivers.groupby('health')['experience_clean'].agg(['count', 'mean', 'median']).round(2)   # стаж по исходу

Стаж известен у 549 погибших водителей, 4 188 раненых и 9 291 не пострадавшего. Средний стаж погибших — 18,61 года, раненых — 14,27: разница больше четырёх лет; медианы 16 и 11. Гипотезы:

- нулевая H0: средний стаж погибших и раненых водителей одинаков, μ₁ = μ₂;
- альтернативная H1: различается, μ₁ ≠ μ₂; альтернатива двусторонняя.

Перед тестом — взгляд на группы. Ящики рядом показывают медианы, квартили и выбросы трёх исходов сразу.

In [ ]:
dead = drivers.loc[drivers['health'] == 'Погиб', 'experience_clean'].dropna()      # стаж погибших водителей
injured = drivers.loc[drivers['health'] == 'Ранен', 'experience_clean'].dropna()   # стаж раненых водителей
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=drivers, x='health', y='experience_clean', order=['Погиб', 'Ранен', 'Не пострадал'], ax=ax)   # ящики трёх исходов
ax.set_title('Стаж водителей по исходу ДТП')
ax.set_xlabel('Исход для водителя')
ax.set_ylabel('Стаж, лет');

Ящик погибших выше: половина погибших водителей — со стажем от 7 до 28 лет, раненых — от 5 до 21 года. Но ящики сильно перекрываются, и стаж от нуля до пятидесяти с лишним лет есть в каждой группе. Разница средних — сдвиг центра, а не разделение групп: среди погибших есть и новички, среди раненых — водители со стажем за сорок лет.

Тест — t-тест двух независимых выборок, `stats.ttest_ind(a, b, equal_var=False)`. Независимые — значит, в группах разные люди и пар между ними нет. Статистика t — разница средних, измеренная в стандартных ошибках:

t = (x̄₁ − x̄₂) / √(s₁² / n₁ + s₂² / n₂),

где x̄ — средние групп, s — их стандартные отклонения, n — размеры. Параметр `equal_var=False` выбирает вариант Уэлча: дисперсии групп не предполагаются равными, и каждая группа входит в стандартную ошибку со своим разбросом. По умолчанию `equal_var=True` — классический t-тест Стьюдента, который считает разброс групп одинаковым; в курсе везде t Уэлча, почему — показывает ловушка в конце раздела. Нормальность стажа отвергнута на семинаре 11, но в каждой группе сотни значений, и по центральной предельной теореме средние распределены почти нормально.

In [ ]:
welch = stats.ttest_ind(dead, injured, equal_var=False)                  # t-тест Уэлча
welch

t 7,31, p 7,6·10⁻¹³ — отвергаем: средний стаж погибших и раненых водителей различается. Поле `df` — число степеней свободы по формуле Уэлча, дробное, около 670: оно ближе к размеру меньшей группы, чем к сумме размеров, потому что точность разницы ограничивает малая группа.

Размер эффекта. p говорит, что разница не случайна, но не говорит, велика ли она. Мера величины для двух средних — d Коэна: разница средних в стандартных отклонениях,

d = (x̄₁ − x̄₂) / s, где s = √((s₁² + s₂²) / 2),

s — объединённое отклонение двух групп, корень из среднего их дисперсий. Функции для d в scipy нет: d считается одной строкой. Знак d показывает направление: d больше нуля, если первая группа в среднем больше.

In [ ]:
pooled = np.sqrt((dead.var() + injured.var()) / 2)                       # объединённое отклонение
d = (dead.mean() - injured.mean()) / pooled                              # d Коэна
print(round(dead.mean() - injured.mean(), 2), round(pooled, 2), round(d, 2))

Разница 4,33 года при отклонении 12,58 — треть отклонения: d 0,34. Принятые пороги Коэна: 0,2 — малый эффект, 0,5 — средний, 0,8 — большой; 0,34 — малый, ближе к порогу 0,2, чем к 0,5. Вывод словами: средний стаж погибших водителей на 4,3 года больше, чем раненых (t-тест Уэлча, n = 549 и 4 188, t = 7,31, p < 0,001, d = 0,34 — малый эффект).

Оговорка о причине. Тест говорит о связи, а не о причине: опыт не убивает. Водители с большим стажем старше, а с возрастом те же травмы тяжелее; опытные чаще ездят по трассам, где выше скорость. Сравнение двух групп не уравнивает их ни по возрасту, ни по дорогам, поэтому в выводе — «погибшие водители опытнее», а не «опыт опасен».

Ловушка без ошибки — t-тест без `equal_var=False`. Другой вопрос: отличается ли стаж водителей, не предоставивших преимущество пешеходу, от стажа водителей без нарушений? Первое нарушение ПДД участника — столбец `violation`.

In [ ]:
violators = drivers.loc[drivers['violation'] == 'Непредоставление преимущества в движении пешеходу', 'experience_clean'].dropna()   # не уступили пешеходу
lawful = drivers.loc[drivers['violation'] == 'Нет нарушений', 'experience_clean'].dropna()   # водители без нарушений
student = stats.ttest_ind(violators, lawful)                             # по умолчанию — Стьюдент
print(len(violators), len(lawful), round(violators.std(), 1), round(lawful.std(), 1))
print(round(student.pvalue, 3), round(stats.ttest_ind(violators, lawful, equal_var=False).pvalue, 3))

37 водителей с этим нарушением против 6 583 без нарушений, отклонения стажа 13,9 и 11,7 года. Тест Стьюдента даёт p 0,048 — «отвергаем», тест Уэлча — p 0,103, «не отвергаем». Стьюдент объединяет дисперсии с весами по размерам групп: большая группа с меньшим разбросом подменяет разброс маленькой, стандартная ошибка разницы занижена, и p слишком мало. Когда группы разного размера и разброса, верен Уэлч; когда одинаковые, оба теста дают почти одно и то же. Поэтому в курсе всегда `equal_var=False`: при одинаковом разбросе Уэлч почти ничего не теряет, а при разном — спасает от ложного «отвергаем».

## 2. Непараметрический двойник: Манна–Уитни

t-тест сравнивает средние, а у стажа правый хвост: среднее тянут водители с большим стажем, а на малых группах один выброс сдвигает среднее заметно. Непараметрический двойник t-теста для двух независимых групп — критерий Манна–Уитни, `stats.mannwhitneyu(a, b)`. Он ставит значения обеих групп в один ряд по возрастанию и нумерует — это ранги — и считает статистику U: в скольких парах «значение из первой группы, значение из второй» первое больше; равные значения идут за половину пары. Нулевая гипотеза: обе группы взяты из одного распределения, и в случайной паре каждая группа больше с вероятностью 1/2. Пар всего n₁ · n₂, поэтому U / (n₁ · n₂) — доля пар, где погибший водитель опытнее раненого, если ничьи считать за половину пары. Сколько пар с равным стажем, считает произведение `value_counts()` двух групп: для каждого стажа — число погибших, умноженное на число раненых.

In [ ]:
mw = stats.mannwhitneyu(dead, injured)                                   # критерий Манна–Уитни
n_pairs = len(dead) * len(injured)                                       # все пары «погибший, раненый»
ties = (dead.value_counts() * injured.value_counts()).sum()              # пары с равным стажем
print(mw.statistic, mw.pvalue, round(mw.statistic / n_pairs, 3))         # U, p и доля пар
print(round((mw.statistic - ties / 2) / n_pairs, 3), round(ties / n_pairs, 3))   # опытнее и поровну

U 1 377 126, p 4,2·10⁻¹⁴ — отвергаем, как и t-тест. Погибший водитель опытнее раненого в 58,6 % пар, в 2,6 % пар стаж одинаков, и такая ничья идёт за половину пары: U / (n₁ · n₂) = 0,586 + 0,026 / 2 = 0,599. Если бы группы не различались, доля была бы 0,5. Это понятная юристу запись размера разницы. Выбросы на ранги почти не влияют: самый большой стаж — просто последний ранг, сколько бы лет в нём ни было.

Какой тест выбрать:

| Данные | Тест |
|---|---|
| нормальность не отвергнута или в каждой группе не меньше 30 значений, вопрос о среднем | t Уэлча, `stats.ttest_ind(a, b, equal_var=False)` |
| сильная асимметрия или выбросы при малых группах | Манна–Уитни, `stats.mannwhitneyu(a, b)` |
| порядковая шкала: баллы, степени тяжести, оценки | Манна–Уитни: у таких значений нет осмысленного среднего |

Тест выбирают по вопросу и данным до расчёта, а не по тому, у какого p меньше. На больших выборках тесты обычно согласны, но отвечают на разные вопросы: t — о средних, Манна–Уитни — о том, какая группа чаще больше.

Второй вопрос — возраст машин у женщин и мужчин за рулём. Столбец `gender` — пол участника.

In [ ]:
women = drivers.loc[drivers['gender'] == 'Женский', 'vehicle_age'].dropna()   # возраст машин женщин
men = drivers.loc[drivers['gender'] == 'Мужской', 'vehicle_age'].dropna()      # возраст машин мужчин
age_test = stats.ttest_ind(women, men, equal_var=False)                  # t-тест Уэлча
d_age = (women.mean() - men.mean()) / np.sqrt((women.var() + men.var()) / 2)   # d Коэна
print(len(women), len(men), round(women.mean(), 2), round(men.mean(), 2), women.median(), men.median())
print(age_test.pvalue, stats.mannwhitneyu(women, men).pvalue, round(d_age, 2))

У женщин машины в среднем моложе почти на год, 10,69 против 11,63; медианы одинаковые — по 10 лет. t Уэлча: p 3,7·10⁻⁸, Манна–Уитни: p 2,0·10⁻⁴ — оба отвергают. Но d −0,13 по модулю меньше порога малого эффекта 0,2: разница ничтожна. Значимой её делают 14 тысяч водителей, а не величина. Значимо — не значит важно: для вывода юриста год разницы в среднем возрасте машины при одинаковых медианах почти ничего не говорит. Минус у d — направление: первая группа, женщины, меньше.

## 3. Парные данные

Вопрос: изменилось ли число ДТП в районах Брянской области за десять лет, с 2015 по 2025 год? Наблюдение здесь — район, и у каждого района два числа: ДТП в 2015 году и в 2025-м. Это парные данные: значения связаны ключом — районом, и каждый район сравнивается сам с собой. Парный t-тест, `stats.ttest_rel(a, b)`, берёт разности внутри пар и проверяет, что их среднее равно нулю: общий уровень района — много в нём ДТП или мало — из разности вычитается, остаётся изменение.

Берутся районы, где в 2015 году было не меньше 20 ДТП: в маленьких районах счётчики слишком малы. Первая попытка — две группировки по годам.

In [ ]:
year = dtp['datetime'].dt.year                                           # год каждого ДТП
first = dtp[year == 2015].groupby('district').size()                     # ДТП по районам, 2015
first = first[first >= 20]                                               # районы от 20 ДТП
last = dtp[year == 2025].groupby('district').size()                      # ДТП по районам, 2025
print(len(first), len(last))

16 значений в первом ряду и 29 во втором. Ошибка в следующей ячейке намеренная.

In [ ]:
stats.ttest_rel(first, last)                                             # парный t-тест

`ValueError: Array shapes are incompatible for broadcasting.`: в первом ряду 16 значений, во втором 29, и пар из них не составить — отбор «от 20 ДТП» применён только к 2015 году. Хуже, когда длины совпадают случайно: тогда `ttest_rel` молча сложит пары по позициям, а не по названиям районов, и ответ будет неверным без всякой ошибки. Парному тесту нужна таблица, выровненная по ключу: строка — район, столбцы — годы. Такую таблицу строит `pd.crosstab` района и года — таблица сопряжённости с семинара 5; у района без ДТП в каком-то году в ней ноль, а не пропуск. Четыре строки с наибольшим числом ДТП в 2015 году:

In [ ]:
by_year = pd.crosstab(dtp['district'], year)                             # ДТП: район × год
by_year[[2015, 2025]].nlargest(4, 2015)                                  # четыре самых больших

Первая строка — не район. «Брянская область» — 3 615 ДТП без указания района, и по координатам почти все они — в самом Брянске, который в таблице отдельным значением не выделен; подробнее — в описании данных `datasets/legal/README.md`. В 2015 году у этой строки 470 ДТП, почти втрое больше, чем в Брянском районе, самом большом, и к 2025 году — на 373 меньше. Город с районами не сравнивают: движение в нём другое, а одна такая разность раздувает разброс разностей и сама решает, что покажет тест. Поэтому строку убирает `drop`, и пары составляют только районы. Рядом с парным t — тест Вилкоксона для пар, знаковый ранговый тест семинара 12 по разностям пар, `stats.wilcoxon(a, b)`: его нулевая гипотеза — разности симметричны относительно нуля.

In [ ]:
by_year = by_year.drop('Брянская область')                               # районы без Брянска
pairs = by_year.loc[by_year[2015] >= 20, [2015, 2025]]                   # районы от 20 ДТП
paired = stats.ttest_rel(pairs[2015], pairs[2025])                       # парный t-тест
unpaired = stats.ttest_ind(pairs[2015], pairs[2025], equal_var=False)    # пары забыты
signed = stats.wilcoxon(pairs[2015], pairs[2025])                        # Вилкоксон для пар
print(len(by_year), len(pairs), pairs.mean().round(1).tolist())
print((pairs[2015] - pairs[2025]).agg(['min', 'median', 'max']).tolist())   # снижение по районам
print(paired.pvalue, unpaired.pvalue, signed.pvalue)

Районов 28, из них с 20 и более ДТП в 2015 году — 15: в среднем 50,9 ДТП в 2015 году и 19,7 в 2025-м. Снижение — во всех 15 районах, от 4 ДТП до 104, медиана 29. Парный тест: p 1,5·10⁻⁴ — отвергаем. Тот же расчёт непарным t Уэлча, как если бы 2015 и 2025 годы были разными районами: p 0,0074 — тоже отвергаем, но p почти в 50 раз больше. Пара даёт силу: районы различаются в разы, и непарный тест видит этот разброс как шум. Районы, большие в 2015 году, остались большими и в 2025-м, поэтому в разностях уровень района вычитается — не до конца: у больших районов и снижение больше, — и разброс разностей меньше того шума, который видит непарный тест. Тест Вилкоксона для пар: p 6,5·10⁻⁴ — отвергаем; рядом с парным t его ставят, когда разности скошены или в них выброс.

Оговорка к выводу: снижение во многом объясняется учётом — лёгких ДТП в таблице 576 в 2015 году и 46 в 2025-м, об этом — описание данных `datasets/legal/README.md`. Тест говорит, что ДТП в районах стало меньше, но не почему.

## 4. Больше двух групп

Исходов для водителя три, и вопрос «различается ли стаж по исходу» — о трёх группах сразу. Три попарных t-теста — три проверки, и вероятность хотя бы одной случайной значимости уже больше 5 %. Тест для нескольких групп сразу — дисперсионный анализ, ANOVA, `stats.f_oneway(a, b, c, equal_var=False)`. Нулевая гипотеза: средние всех групп равны. Статистика F — отношение разброса между средними групп к разбросу внутри групп: чем дальше средние друг от друга по сравнению с разбросом внутри, тем больше F. Параметр `equal_var=False` выбирает ANOVA Уэлча: как и у t-теста, дисперсии групп не предполагаются равными. По умолчанию scipy считает классическую ANOVA с общей дисперсией, а у неё та же слабость, что у t Стьюдента: на группах разного размера и разброса — здесь 549, 4 188 и 9 291 водитель — она может ошибаться, поэтому в курсе и здесь Уэлч. Непараметрический двойник — критерий Крускала–Уоллиса, `stats.kruskal(a, b, c)`, по рангам.

Отвергнутая гипотеза ANOVA значит только «не все средние равны». Какие группы различаются, показывают попарные сравнения с поправкой Бонферрони: три пары — порог 0,05 / 3.

In [ ]:
unhurt = drivers.loc[drivers['health'] == 'Не пострадал', 'experience_clean'].dropna()   # стаж не пострадавших
print(stats.f_oneway(dead, injured, unhurt, equal_var=False))            # ANOVA Уэлча
print(stats.kruskal(dead, injured, unhurt))                              # Крускал–Уоллис
for name, a, b in [('погибшие и раненые', dead, injured), ('погибшие и не пострадавшие', dead, unhurt),
                   ('раненые и не пострадавшие', injured, unhurt)]:
    p = stats.ttest_ind(a, b, equal_var=False).pvalue                     # попарный t Уэлча
    print(name, p, 'отвергаем' if p < 0.05 / 3 else 'не отвергаем')

F 64,3 и H 160,7, оба p < 0,001: стаж по исходам различается. После поправки различаются все три пары: погибшие опытнее и раненых, и не пострадавших, а раненые — наименее опытные. Поправка Бонферрони — самая простая и самая строгая; поправки мягче, Холма и Бенджамини–Хохберга, за рамками курса.

## 5. Две категории: хи-квадрат

Вопрос: чаще ли ДТП в тёмное время суток заканчиваются гибелью людей? Оба признака категориальные: тёмное время — да или нет, по столбцу `light`, где тёмное время записано словом «темное»; ДТП с погибшими — да или нет, `dead_count` больше нуля. Нулевая гипотеза: признаки независимы, доля ДТП с погибшими одинакова в светлое и тёмное время; альтернативная — связаны. Данные для теста — таблица сопряжённости, счётчики ДТП по сочетаниям значений; её строит `pd.crosstab`.

In [ ]:
dark = dtp['light'].str.contains('темное').rename('тёмное время')       # тёмное время суток
fatal = (dtp['dead_count'] > 0).rename('с погибшими')                    # ДТП с погибшими
table = pd.crosstab(dark, fatal)                                         # таблица сопряжённости
table

Тест — хи-квадрат независимости, `stats.chi2_contingency(таблица)`. Он сравнивает наблюдаемые счётчики с ожидаемыми при независимости: ожидаемое в клетке — сумма строки, умноженная на сумму столбца и делённая на число всех наблюдений. Статистика

χ² = Σ (наблюдаемое − ожидаемое)² / ожидаемое,

сумма по всем клеткам; степеней свободы (строк − 1) · (столбцов − 1). У таблицы 2 × 2 scipy по умолчанию вносит поправку Йейтса на непрерывность, она немного уменьшает χ²; для p в курсе её оставляют. Результат — объект с полями `statistic`, `pvalue`, `dof` и `expected_freq` — таблицей ожидаемых.

In [ ]:
chi = stats.chi2_contingency(table)                                      # хи-квадрат независимости
print(round(chi.statistic, 1), chi.pvalue, chi.dof)
chi.expected_freq.round(1)                                               # ожидаемые при независимости

χ² 161,6 при одной степени свободы, p 5,0·10⁻³⁷ — отвергаем: время суток и гибель людей связаны. При независимости ДТП с погибшими в тёмное время было бы около 468, наблюдается 676.

Размер связи — V Крамера:

V = √(χ² / (n · (k − 1))),

где n — число наблюдений в таблице, k — меньшее из чисел строк и столбцов. χ² для V берут без поправки Йейтса, `stats.chi2_contingency(таблица, correction=False)`, как в учебниках, — то же V возвращает `stats.contingency.association(таблица)`, — а p остаётся с поправкой. V от 0 — независимость — до 1 — один признак полностью определяет другой. Юристу понятнее доли по строкам: доля ДТП с погибшими в светлое и в тёмное время, `pd.crosstab(..., normalize='index')`.

In [ ]:
plain = stats.chi2_contingency(table, correction=False)                  # χ² без поправки Йейтса
v = np.sqrt(plain.statistic / (table.to_numpy().sum() * (min(table.shape) - 1)))   # V Крамера
print(round(v, 3))
shares = pd.crosstab(dark, fatal, normalize='index')                     # доли по строкам
shares.round(3)

V 0,127 — малая связь: пороги для V — 0,1 малая, 0,3 средняя, 0,5 большая. Доли говорят ярче: в тёмное время ДТП с погибшими 19,4 %, в светлое — 10,3 %, почти вдвое больше. Это и пишут в выводе: в тёмное время суток доля ДТП с погибшими почти вдвое выше, чем в светлое (хи-квадрат, n = 10 114, χ² = 161,6, p < 0,001, V = 0,13). Малый V при двукратной разнице долей — не противоречие: V мерит связь по всем ДТП, а большинство ДТП в любое время суток обходится без погибших. Оговорка о причине та же: в темноте выше скорость на трассах, хуже видны пешеходы, больше нетрезвых водителей; тест называет связь, а не причину.

Тёмное время в таблице — три значения `light` из пяти, а сумерки попали вместе с днём в светлое время. Разбивка по всем пяти значениям показывает, где связь сильнее.

In [ ]:
by_light = pd.crosstab(dtp['light'], fatal, normalize='index')[True].sort_values()   # доля ДТП с погибшими
fig, ax = plt.subplots(figsize=(8, 4))
by_light.plot.barh(ax=ax)                                                # столбцы по освещению
ax.set_title('Доля ДТП с погибшими по освещению')
ax.set_xlabel('Доля ДТП с погибшими')
ax.set_ylabel('')
by_light.round(3)

При включённом освещении доля ДТП с погибшими — 0,121, почти как днём, 0,101. Без освещения — 0,286, почти втрое больше, чем днём; при невключённом — 0,252. Связь «темнота — гибель» во многом связь «неосвещённая дорога — гибель», и для юриста это вопрос к содержанию дорог. Но и это связь, а не причина: неосвещены в основном загородные трассы, где выше скорость.

Ловушка без ошибки — доли вместо счётчиков. Таблица долей только что посчитана, и её легко передать в тест:

In [ ]:
stats.chi2_contingency(shares)                                           # доли вместо счётчиков

χ² 0, p 1 — «не отвергаем», и ни ошибки, ни предупреждения: тест получил числа меньше единицы вместо тысяч ДТП, а поправка Йейтса на таких числах обнуляет статистику. Тесту нужны счётчики, а не доли. А таблица с текстом — сами столбцы `dtp[['light', 'severity']]` вместо таблицы сопряжённости или таблица после `reset_index()` с названиями строк в столбце — остановит scipy ошибкой `TypeError: '<' not supported between instances of 'str' and 'int'`: функция ждёт числа, а получает слова. Таблицу счётчиков строит `pd.crosstab`.

При запасе времени — отношение шансов. Шанс — не доля, а отношение «с погибшими к без погибших»: в тёмное время 676 к 2 809, в светлое — 683 к 5 946. Отношение шансов показывает, во сколько раз шанс в одной строке больше, чем в другой; его же возвращает полем `statistic` точный тест Фишера, о нём ниже.

In [ ]:
odds = (table.loc[True, True] / table.loc[True, False]) / (table.loc[False, True] / table.loc[False, False])   # отношение шансов
print(round(odds, 2), round(stats.fisher_exact(table).statistic, 2))

Шанс ДТП с погибшими в тёмное время в 2,1 раза выше, чем в светлое. Отношение шансов не меняется, если поменять строки и столбцы местами: шансы гибели в темноте против дня и шансы темноты у ДТП с погибшими против ДТП без погибших дают одно и то же число.

Второй вопрос — связь, которой не видно: погода и тяжесть ДТП. Четыре частые погоды — ясно, пасмурно, дождь, снегопад — и три тяжести; туман, метель и жара дали бы клетки с ожидаемыми меньше пяти, а запись «Без пострадавших (нет данных)» в таблице одна.

In [ ]:
four = dtp[dtp['weather'].isin(['Ясно', 'Пасмурно', 'Дождь', 'Снегопад']) & (dtp['severity'] != 'Без пострадавших (нет данных)')]   # частые погоды
weather = pd.crosstab(four['weather'], four['severity'])                 # погода × тяжесть
chi_w = stats.chi2_contingency(weather)                                  # хи-квадрат независимости
v_w = np.sqrt(chi_w.statistic / (weather.to_numpy().sum() * (min(weather.shape) - 1)))   # V Крамера
print(round(chi_w.statistic, 1), round(chi_w.pvalue, 2), chi_w.dof, round(v_w, 3), round(chi_w.expected_freq.min(), 1))

χ² 4,6 при шести степенях свободы, p 0,59 — не отвергаем; V 0,015. Доли тяжёлых ДТП и ДТП с погибшими в дождь, снегопад и ясную погоду почти одинаковы. «Не отвергаем» не значит «погода не влияет»: связь погоды с тяжестью на этих данных не видна, а погода может влиять на число ДТП, о котором тест не спрашивает. Наименьшее ожидаемое — 31,6: правило хи-квадрата — в каждой клетке ожидаемых не меньше пяти — выполнено.

Малые таблицы. Когда ожидаемые меньше пяти, хи-квадрат неточен: его p опирается на приближение, которое на малых счётчиках не работает. Для таблицы 2 × 2 есть точный тест Фишера, `stats.fisher_exact(таблица)`: p — сумма вероятностей всех таблиц с теми же суммами строк и столбцов, которые не вероятнее наблюдаемой; таблицы перебираются все, без приближения. Пример — Злынковский район, 69 ДТП.

In [ ]:
small = dtp[dtp['district'] == 'Злынковский район']                      # ДТП одного района
small_table = pd.crosstab(small['light'].str.contains('темное'), small['dead_count'] > 0)   # таблица 2 × 2
small_chi = stats.chi2_contingency(small_table)                          # хи-квадрат на малой таблице
print(small_table.to_numpy().tolist(), round(small_chi.expected_freq.min(), 2))
print(round(stats.fisher_exact(small_table).pvalue, 2), round(small_chi.pvalue, 2))   # Фишер и хи-квадрат

Ожидаемых ДТП с погибшими в тёмное время — 3,65, меньше пяти: хи-квадрат здесь не годится, хотя выдаёт p 0,91 без предупреждения. Фишер считает точно: p 0,73 — не отвергаем. На 69 ДТП связь, которую область показывает на 10 тысячах, не видна: малая выборка слаба. В больших таблицах с редкими категориями редкие значения объединяют — как здесь три вида тёмного времени объединены в одно.

## 6. Значимость корреляции

Коэффициент корреляции Пирсона r с семинара 9 — мера линейной связи двух числовых столбцов, от −1 до 1. Тест к нему: нулевая гипотеза — связи нет, r = 0. Функция `stats.pearsonr(x, y)` возвращает r и p; `stats.spearmanr(x, y)` — коэффициент Спирмена ρ: корреляцию рангов, для монотонной, не обязательно линейной связи, устойчивую к выбросам. Вопрос: связаны ли стаж водителя и возраст его машины? Пары берутся там, где известны оба значения: `dropna()` по двум столбцам сразу.

In [ ]:
pair = drivers[['experience_clean', 'vehicle_age']].dropna()             # водители с обоими значениями
pearson = stats.pearsonr(pair['experience_clean'], pair['vehicle_age'])   # корреляция Пирсона
spearman = stats.spearmanr(pair['experience_clean'], pair['vehicle_age'])   # корреляция Спирмена
print(len(pair), round(pearson.statistic, 3), pearson.pvalue)
print(round(spearman.statistic, 3), spearman.pvalue)

r −0,030, p 3,7·10⁻⁴ — отвергаем: связь «значима». Но r² = 0,0009: стаж объясняет меньше десятой доли процента разброса возраста машины. Это ловушка вывода: «корреляция значима» звучит как «связь есть», а на 13 900 парах значим любой r больше 0,017 по модулю. Спирмен −0,076 — тоже ничтожно. Вывод словами: связь стажа и возраста машины статистически значима, но ничтожна (корреляция Пирсона, n = 13 900, r = −0,03, p < 0,001); на практике её нет.

Ловушка без ошибки — корреляция по столбцам с пропусками:

In [ ]:
stats.pearsonr(drivers['experience_clean'], drivers['vehicle_age'])      # столбцы с пропусками

`nan` без ошибки и без предупреждения: как и тесты семинара 12, `pearsonr` пропуски не пропускает. Метод pandas `drivers['experience_clean'].corr(drivers['vehicle_age'])` пропуски пропускает сам и даёт те же −0,030, но без p. Пары отбирают по двум столбцам сразу: `dropna()` у каждого столбца отдельно даёт ряды разной длины, и пары разъезжаются.

Размер эффекта — сводная таблица порогов:

| Мера | Для чего | ничтожный | малый | средний | большой |
|---|---|---|---|---|---|
| d Коэна | разница двух средних | меньше 0,2 | от 0,2 | от 0,5 | от 0,8 |
| V Крамера | связь двух категорий | меньше 0,1 | от 0,1 | от 0,3 | от 0,5 |
| r Пирсона, ρ Спирмена | связь двух числовых столбцов | меньше 0,1 | от 0,1 | от 0,3 | от 0,5 |

Пороги берутся по модулю: d −0,4 и 0,4 — одинаково малые. Пороги V — для таблиц, где у одного из признаков два значения, как у тёмного времени; для больших таблиц Коэн предлагал пороги ниже, делённые на √(k − 1), поэтому V таблиц разной формы между собой не сравнивают. Пороги условны: их предложил Коэн для психологии, и важность для юриста решает не таблица, а дело: двукратная разница в доле ДТП с погибшими важна и при малом V. В разведочном анализе этапа 2 у корреляции другая шкала — слабая до 0,4, средняя от 0,4, сильная от 0,7: она описывает, какие столбцы таблицы связаны заметнее других, а пороги Коэна — размер эффекта проверенной гипотезы, и в этапе 3 пишут их. Сегодняшние эффекты: d 0,34 и V 0,127 — малые; d −0,13, V 0,015 и r −0,03 — ничтожные.

## 7. Честный вывод и этап 3

Сводная таблица гипотез — образец раздела отчёта этапа 3 на данных Брянской области: шесть разных вопросов, на каждый — один тест, строка на гипотезу. Один вопрос — одна гипотеза: стаж погибших и раненых проверен t-тестом Уэлча, а критерий Манна–Уитни на тот же вопрос в таблицу не идёт — тест выбирают до расчёта, а не по тому, у какого p меньше. Нормальности стажа в таблице тоже нет: это проверка условия, по которой выбирают тест, а не вопрос о деле, и в семейство поправки она не входит. Четыре гипотезы посчитаны выше, две новые — стаж женщин и мужчин за рулём, t Уэлча, и недостатки дорожных условий против ДТП с погибшими, хи-квадрат; недостатки установлены, если `road_defects` больше нуля.

In [ ]:
exp_women = drivers.loc[drivers['gender'] == 'Женский', 'experience_clean'].dropna()   # стаж женщин за рулём
exp_men = drivers.loc[drivers['gender'] == 'Мужской', 'experience_clean'].dropna()     # стаж мужчин за рулём
gender_test = stats.ttest_ind(exp_women, exp_men, equal_var=False)       # t-тест Уэлча
d_gender = (exp_women.mean() - exp_men.mean()) / np.sqrt((exp_women.var() + exp_men.var()) / 2)   # d Коэна
defects = (dtp['road_defects'] > 0).rename('недостатки дороги')          # недостатки дорожных условий
defect_table = pd.crosstab(defects, fatal)                               # таблица сопряжённости
chi_d = stats.chi2_contingency(defect_table)                             # хи-квадрат независимости
plain_d = stats.chi2_contingency(defect_table, correction=False)         # χ² без поправки Йейтса
v_d = np.sqrt(plain_d.statistic / (defect_table.to_numpy().sum() * (min(defect_table.shape) - 1)))   # V Крамера
print(round(exp_women.mean(), 1), round(exp_men.mean(), 1), round(d_gender, 3))
print(round(chi_d.statistic, 2), round(chi_d.pvalue, 3), round(v_d, 3))

Стаж женщин за рулём в среднем 8,4 года, мужчин — 17,1: d −0,872, большой эффект. Недостатки дороги и гибель людей: χ² 1,51, p 0,219 — не отвергаем, V 0,013.

Таблица — строка на гипотезу: вопрос, тест, n, статистика, p, размер эффекта числом и словом по порогам раздела 6, решение при 0,05 и после поправки. У парного t в столбце «размер» прочерк: размера эффекта для пар в курсе нет, в вывод идёт само снижение — в среднем на 31 ДТП на район. Проверок шесть, больше пяти, — поэтому решение записано и с поправкой Бонферрони, при пороге 0,05 / 6.

In [ ]:
report = pd.DataFrame([
    {'вопрос': 'стаж погибших и раненых водителей', 'тест': 't Уэлча', 'n': len(dead) + len(injured),
     'статистика': welch.statistic, 'p': welch.pvalue, 'эффект': d, 'размер': 'малый'},
    {'вопрос': 'ДТП в районах в 2015 и 2025 годах', 'тест': 'парный t', 'n': len(pairs),
     'статистика': paired.statistic, 'p': paired.pvalue, 'эффект': np.nan, 'размер': '—'},
    {'вопрос': 'тёмное время и ДТП с погибшими', 'тест': 'хи-квадрат', 'n': table.to_numpy().sum(),
     'статистика': chi.statistic, 'p': chi.pvalue, 'эффект': v, 'размер': 'малый'},
    {'вопрос': 'стаж и возраст машины', 'тест': 'Пирсон', 'n': len(pair),
     'статистика': pearson.statistic, 'p': pearson.pvalue, 'эффект': pearson.statistic, 'размер': 'ничтожный'},
    {'вопрос': 'стаж женщин и мужчин за рулём', 'тест': 't Уэлча', 'n': len(exp_women) + len(exp_men),
     'статистика': gender_test.statistic, 'p': gender_test.pvalue, 'эффект': d_gender, 'размер': 'большой'},
    {'вопрос': 'недостатки дороги и ДТП с погибшими', 'тест': 'хи-квадрат', 'n': defect_table.to_numpy().sum(),
     'статистика': chi_d.statistic, 'p': chi_d.pvalue, 'эффект': v_d, 'размер': 'ничтожный'},
], index=range(1, 7))
report['решение'] = np.where(report['p'] < 0.05, 'отвергаем', 'не отвергаем')                 # решение при 0,05
report['после поправки'] = np.where(report['p'] < 0.05 / len(report), 'отвергаем', 'не отвергаем')   # порог 0,05 / 6
report.round(3)

При 0,05 отвергнуты пять нулевых гипотез из шести, после поправки — те же пять: их p намного меньше порога 0,05 / 6 = 0,0083. p, выведенные нулём, в отчёте записываются как «p < 0,001». Не отвергнута одна: связь недостатков дороги с гибелью людей на этих данных не видна. Столбцы «эффект» и «размер» отвечают на вопрос части 7 этапа 3 — какие различия значимы и при этом важны по величине: большой эффект у стажа женщин и мужчин, d −0,872; малые — у стажа погибших и раненых, d 0,345, и у тёмного времени, V 0,127; корреляция стажа с возрастом машины значима только из-за размера выборки.

Чек-лист честного вывода — пять ошибок, которые делают вывод сильнее данных:

1. **Альтернатива после данных.** Направление альтернативы выбирают из вопроса до расчёта, по умолчанию она двусторонняя; односторонняя, выбранная по данным, на деле удваивает уровень значимости.
2. **Много тестов без поправки.** Если все шесть нулевых гипотез верны, при 0,05 хотя бы одна случайная «значимость» выпадет с вероятностью около четверти; больше пяти гипотез — поправка Бонферрони, а в отчёт идут все проверенные гипотезы, не только значимые. Один вопрос — одна гипотеза: два теста на один вопрос — тоже лишняя проверка.
3. **Значимость как важность.** p говорит, что разница не случайна, размер эффекта — велика ли она. Рядом с p пишут d, V или r и саму разницу словами: «на 4,3 года», «почти вдвое».
4. **Связь как причина.** Сравнение групп и таблица сопряжённости показывают связь. Погибшие водители опытнее не потому, что опыт опасен; темнота связана с гибелью и через скорость, и через трезвость. Причину называют, только если группы различаются одним этим признаком, а в данных наблюдений так почти не бывает.
5. **Парное как непарное.** Одни и те же районы, суды или люди в два момента времени — парные данные: парный тест по таблице, выровненной по ключу. Непарный тест на них теряет силу, а невыровненные ряды дают ошибку или, хуже, молча неверные пары.

### Этап 3 и контрольная работа № 2

Гипотезы 3 и 4 этапа 3 — сегодняшние: сравнение двух групп и связь двух категориальных столбцов. Запись — по формату семинара 12. Для гипотезы 3: размеры и средние или медианы групп, тест по нормальности и размеру групп — t Уэлча или Манна–Уитни, d Коэна, ящики групп и оговорка о причине. Для гипотезы 4: таблица долей по строкам, хи-квадрат или тест Фишера для малой таблицы, V Крамера и вывод словами по долям. Функции `t_two`, `mw_two`, `chi_square`, `fisher_2x2` и `effect_label` из упражнений — готовые инструменты этих гипотез. Этап 3 сдаётся на семинаре 14, файл `Группа-Фамилия-Имя-project-03.ipynb`; задание — [project/03-hypotheses.md](../project/03-hypotheses.md).

На семинаре 14 — контрольная работа № 2: визуализация, предобработка и гипотезы, семинары 8–13; сегодняшние тесты в неё входят. Как готовиться — в домашнем задании.

## Итоги

Две независимые группы — t Уэлча `stats.ttest_ind(a, b, equal_var=False)`: по умолчанию scipy считает t Стьюдента, и при группах разного размера и разброса он ошибается. Размер разницы — d Коэна: разница средних, делённая на √((s₁² + s₂²) / 2). Непараметрический двойник — Манна–Уитни `stats.mannwhitneyu(a, b)`: U / (n₁ · n₂) — доля пар, где первая группа больше, ничья — за половину пары. Значимо — не значит важно: на тысячах наблюдений и d −0,13 даёт p < 0,001. Парные данные выравнивают по ключу — `pd.crosstab` района и года — и проверяют парным тестом `stats.ttest_rel(a, b)` или Вилкоксоном `stats.wilcoxon(a, b)`; ряды разной длины дают `ValueError`, а совпавшие по длине, но не выровненные — молча неверный ответ. Больше двух групп — ANOVA Уэлча `stats.f_oneway(..., equal_var=False)` и Крускал–Уоллис `stats.kruskal`, затем попарные сравнения с поправкой Бонферрони. Две категории — таблица сопряжённости `pd.crosstab` и хи-квадрат `stats.chi2_contingency`: тесту нужны счётчики, а не доли и не текст; ожидаемые не меньше пяти, иначе тест Фишера `stats.fisher_exact`; размер связи — V Крамера √(χ² / (n · (k − 1))) по χ² без поправки Йейтса, а вывод — словами по долям строк. Корреляция — `stats.pearsonr` и `stats.spearmanr` по парам без пропусков: на больших выборках значим и ничтожный r. Размер эффекта по порогам Коэна: d — 0,2, 0,5, 0,8; V и r — 0,1, 0,3, 0,5. Честный вывод: один вопрос — одна гипотеза, альтернатива до данных, поправка на много проверок, размер эффекта рядом с p, связь не причина, парное — парным тестом.

Переходите к упражнениям: откройте `exercises/seminar-13-tasks.ipynb`, формулировки в `exercises/seminar-13-tasks.md`; данные там по Московской области. Функции части Б — инструменты гипотез 3 и 4 этапа 3: t Уэлча с d Коэна, Манна–Уитни, парный тест с выравниванием, ANOVA Уэлча и Крускал–Уоллис, хи-квадрат с V Крамера, тест Фишера, значимость корреляции, размер эффекта словами; в части В из них собираются попарные сравнения с поправкой, проверка категорий против одного столбца и таблица гипотез курса. Необязательная задача — оставшиеся гипотезы этапа 3 на своём датасете. Этап 3 сдаётся на семинаре 14, в тот же день контрольная работа № 2.